# Day 31 · Agent 范式与工具协议

**配套讲义**: [`days/day-31.md`](../days/day-31.md) ｜ **本地可跑，不需要 GPU**

定义 7 个客服工具的 schema 并全部 mock 起来，用 20 条测试 query 验证「模型能不能正确选对工具、填对参数」；并说清「让模型输出 JSON」和 「用 function calling」在鲁棒性上的差别。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w6.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. 跑工具自检

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.agent.tools"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 逐个工具看「数据长什么样」

写工具之前先看 mock 数据 —— 这决定了参数设计是否合理。

In [ ]:
import sys; sys.path.insert(0, "..")
from src.agent.tools import MOCK_ORDERS, MOCK_STOCK, RETURN_POLICY

print("订单样本（前 2 条）:")
for oid, o in list(MOCK_ORDERS.items())[:2]:
    print(f"  {oid}: {o}")
print("\n库存样本（前 3 条）:")
for sku, s in list(MOCK_STOCK.items())[:3]:
    print(f"  {sku}: {s}")
print("\n退货政策:", str(RETURN_POLICY)[:200])

## 3. 设计你自己的第 8 个工具

客服场景还缺什么？候选：**改地址 / 催发货 / 申请发票 / 补差价**。
写成一个 `Tool` 并加进注册表 —— **description 要写到你妈能看懂**。

In [ ]:
import sys; sys.path.insert(0, "..")
from src.agent.tools import Tool

my_tool = Tool(
    name="urge_shipping",              # ← 改成你的
    description="当用户催促发货、询问为什么还没发货时使用。需要订单号。",
    parameters={"order_id": {"type": "string", "description": "订单号，形如 A1"}},
    fn=lambda order_id: {"ok": True, "msg": f"{order_id} 已加急"},
)
print("工具名:", my_tool.name)
print("描述长度:", len(my_tool.description), "（太短说明你没写清楚）")
print("参数:", list(my_tool.parameters))

## 验收清单

- [ ] 7 个工具的 schema 能被 `json.dumps` 正常导出（说明结构合法）
- [ ] **幂等性验证通过**：三次 `start_return` 只产生一个退货单
- [ ] 工具幻觉被拦住（模型编造的工具名 → 拒绝执行）
- [ ] 异常输入不崩，返回的是友好话术而不是 traceback
- [ ] 能说清「JSON 输出」vs「function calling」的鲁棒性差异

**卡住了？** 回看 [`days/day-31.md`](../days/day-31.md) 第五节「容易踩的坑」。

> **明天**：`days/day-32.md` —— 多模态 RAG：让用户能「拍图找货」